# Day 2: Intermediate SQL (15 Questions)
## Multiple JOINs

##### 1.Get order_id, customer_city, product_category_name, and item price by joining olist_orders_dataset, olist_customers_dataset, olist_order_items_dataset, and olist_products_dataset.

In [0]:
select orditm.order_id,cus.customer_city,prod.product_category_name,orditm.price 
from brazilian_e_commerce.sql_practice.olist_customers_dataset cus 
join brazilian_e_commerce.sql_practice.olist_orders_dataset ord on cus.customer_id=ord.customer_id
join brazilian_e_commerce.sql_practice.olist_order_items_dataset orditm on orditm.order_id=ord.order_id
join brazilian_e_commerce.sql_practice.olist_products_dataset prod on orditm.product_id=prod.product_id;

##### 2.Find the total freight value paid by customers in each customer_state for orders processed by sellers in olist_sellers_dataset (requires joining orders, customers, items, and sellers).

In [0]:
create table if not exists brazilian_e_commerce.sql_practice.olist_seller_dataset as
select * from read_files("/Volumes/brazilian_e_commerce/sql_practice/brazilian-ecommerce/olist_sellers_dataset.csv");
desc brazilian_e_commerce.sql_practice.olist_seller_dataset;

In [0]:
select cus.customer_state,sum(orditm.freight_value) 
from brazilian_e_commerce.sql_practice.olist_customers_dataset cus 
join brazilian_e_commerce.sql_practice.olist_orders_dataset ord on cus.customer_id=ord.customer_id
join brazilian_e_commerce.sql_practice.olist_order_items_dataset orditm on orditm.order_id=ord.order_id
join brazilian_e_commerce.sql_practice.olist_seller_dataset sell on orditm.seller_id=sell.seller_id
group by cus.customer_state
order by sum(orditm.freight_value) desc;

##### 3.Retrieve order_id, payment method (payment_type), and product category for all completed orders.

In [0]:
create table if not exists brazilian_e_commerce.sql_practice.olist_order_payments_dataset as
select * from read_files("/Volumes/brazilian_e_commerce/sql_practice/brazilian-ecommerce/olist_order_payments_dataset.csv");
desc brazilian_e_commerce.sql_practice.olist_order_payments_dataset;

In [0]:
select * from brazilian_e_commerce.sql_practice.olist_orders_dataset
limit 3;

In [0]:
select orditm.order_id,pymt.payment_type,prod.product_category_name 
from brazilian_e_commerce.sql_practice.olist_order_items_dataset orditm
join brazilian_e_commerce.sql_practice.olist_order_payments_dataset pymt on orditm.order_id = pymt.order_id
join brazilian_e_commerce.sql_practice.olist_products_dataset prod on orditm.product_id = prod.product_id
join brazilian_e_commerce.sql_practice.olist_orders_dataset ord on orditm.order_id = ord.order_id
where ord.order_status = 'delivered';

##### 4.List seller IDs along with the customer_state they most frequently ship to.

In [0]:
select selr.seller_id,cus.customer_state,count(distinct orditm.order_id) as total_orders 
from brazilian_e_commerce.sql_practice.olist_order_items_dataset orditm
join brazilian_e_commerce.sql_practice.olist_seller_dataset selr on orditm.seller_id = selr.seller_id
join brazilian_e_commerce.sql_practice.olist_orders_dataset ord on orditm.order_id = ord.order_id
join brazilian_e_commerce.sql_practice.olist_customers_dataset cus on ord.customer_id = cus.customer_id
group by selr.seller_id,cus.customer_state
order by total_orders desc
limit 10;


## CASE Statements
#####5. Categorize each order into price tiers ('High' if item price > 200, 'Medium' if price between 50 and 200, 'Low' if price < 50) from olist_order_items_dataset.

In [0]:
select order_id,price,
case 
when price > 200 then 'High'
when price between 50 and 200 then 'Medium'
when price < 50 then 'Low'
end as price_category
 from brazilian_e_commerce.sql_practice.olist_order_items_dataset;

##### 6. Create a custom status classification for orders in olist_orders_dataset: 'Completed' for 'delivered', 'In Progress' for 'shipped' or 'approved', and 'Cancelled/Other' for all other statuses. Count orders

In [0]:
select distinct order_status from brazilian_e_commerce.sql_practice.olist_orders_dataset;

In [0]:
select 
case 
when order_status = 'delivered' then 'Completed'
when order_status in ('shipped','approved') then 'In-Progress'
else 'Cancelled/Other'
end as order_status_classification,
count(order_id) as classification_count
from brazilian_e_commerce.sql_practice.olist_orders_dataset
group by order_status_classification;

##### 7. Classify order delivery promptness by comparing order_delivered_customer_date and order_estimated_delivery_date into 'On Time' vs 'Late'.

In [0]:
select * from brazilian_e_commerce.sql_practice.olist_orders_dataset
limit 3;

In [0]:
select order_id,case
when order_delivered_customer_date <= order_estimated_delivery_date then 'On-Time'
else 'Late'
end as delivery_status,
order_status
from brazilian_e_commerce.sql_practice.olist_orders_dataset
where order_status = 'delivered' and order_delivered_customer_date is not null;

#####8. Flag order payment methods as 'Credit' (if payment_type = 'credit_card') or 'Non-Credit' and calculate total transaction value for each type using olist_order_payments_dataset.

In [0]:
select distinct payment_type from brazilian_e_commerce.sql_practice.olist_order_payments_dataset;

In [0]:
select case
when payment_type = 'credit_card' then 'Credit'
else 'Non Credit'
end as payment_category,
round(sum(payment_value),2) as total_transaction_value
from brazilian_e_commerce.sql_practice.olist_order_payments_dataset
group by 1;

## Subqueries
##### 9. Find all orders in olist_orders_dataset whose total price exceeds the average order total across the entire dataset.

In [0]:
with order_totals as (
select order_id,sum(price) as total_ord_price from brazilian_e_commerce.sql_practice.olist_order_items_dataset
group by order_id)
select order_id,total_ord_price from order_totals
where total_ord_price > (select avg(total_ord_price) from order_totals);

##### 10. Identify customers (customer_unique_id) who have placed more orders than the average order count per customer. 

In [0]:
with order_totals as (
    select cus.customer_unique_id,count(ord.order_id) total_order from brazilian_e_commerce.sql_practice.olist_orders_dataset ord
join brazilian_e_commerce.sql_practice.olist_customers_dataset cus on ord.customer_id = cus.customer_id
group by customer_unique_id)
select customer_unique_id,total_order from order_totals
where total_order > (select avg(total_order) from order_totals);

##### 11. Find products in olist_products_dataset that have never been ordered (using WHERE product_id NOT IN (...) or NOT EXISTS).

In [0]:
select product_id,product_category_name from brazilian_e_commerce.sql_practice.olist_products_dataset
where product_id not in (select distinct product_id from brazilian_e_commerce.sql_practice.olist_order_items_dataset);

In [0]:
SELECT 
  (SELECT COUNT(DISTINCT product_id) FROM brazilian_e_commerce.sql_practice.olist_products_dataset) AS product_catalog_count,
  (SELECT COUNT(DISTINCT product_id) FROM brazilian_e_commerce.sql_practice.olist_order_items_dataset) AS ordered_products_count;

In [0]:
select product_id,product_category_name from brazilian_e_commerce.sql_practice.olist_products_dataset prod
where not exists (select 1 from brazilian_e_commerce.sql_practice.olist_order_items_dataset orditm
where prod.product_id = orditm.product_id);


##### 12. List all sellers whose average product price is higher than the overall average product price in olist_order_items_dataset.

In [0]:
with avg_price as (
    select seller_id,avg(price) as seller_avg_price from brazilian_e_commerce.sql_practice.olist_order_items_dataset
    group by seller_id)
select * from avg_price where seller_avg_price > (select avg(price) from brazilian_e_commerce.sql_practice.olist_order_items_dataset);

In [0]:
select seller_id,avg(price) as seller_avg_price from brazilian_e_commerce.sql_practice.olist_order_items_dataset
group by seller_id
having seller_avg_price > (select avg(price) from brazilian_e_commerce.sql_practice.olist_order_items_dataset);

## Combined Concepts
##### 13. Calculate the total revenue and percentage share of revenue generated by each payment_type using a subquery or window function concept with CASE.

In [0]:
-- using subquery
select payment_type, round(sum(payment_value),2) total_payment, round(sum(payment_value) / (select sum(payment_value) from brazilian_e_commerce.sql_practice.olist_order_payments_dataset) * 100,2) as percentage
from brazilian_e_commerce.sql_practice.olist_order_payments_dataset
group by payment_type;

In [0]:
-- window function 
select payment_type, round(sum(payment_value)/ sum(sum(payment_value)) over()*100,2) as percentage
from brazilian_e_commerce.sql_practice.olist_order_payments_dataset
group by payment_type;

##### 14. Identify the top 3 product categories by total sales volume (item count) delivered to the state of 'RJ'.

In [0]:
select prod.product_category_name,count(items.order_item_id)as product_count,cust.customer_state 
from brazilian_e_commerce.sql_practice.olist_products_dataset prod
join brazilian_e_commerce.sql_practice.olist_order_items_dataset items
on prod.product_id = items.product_id
join brazilian_e_commerce.sql_practice.olist_orders_dataset ord
on items.order_id = ord.order_id
join brazilian_e_commerce.sql_practice.olist_customers_dataset cust
on ord.customer_id = cust.customer_id
where cust.customer_state = 'RJ' and ord.order_status = 'delivered'
group by product_category_name,customer_state
order by product_count desc
LIMIT 3;

##### 15. For each customer state, label states as 'High Volume' if they have over 10,000 orders, otherwise 'Standard Volume', and display total revenue per group.

In [0]:
with total_order_cust_state as (
select cust.customer_state,count(orditm.order_item_id) tot_order,round(sum(orditm.price),2) tot_revenue from brazilian_e_commerce.sql_practice.olist_order_items_dataset orditm
join brazilian_e_commerce.sql_practice.olist_orders_dataset ord on orditm.order_id = ord.order_id
join brazilian_e_commerce.sql_practice.olist_customers_dataset cust on ord.customer_id = cust.customer_id
group by customer_state)
select case 
when tot_order > 10000 then 'High Volume'
else 'Standard Volume'
end as volume_segment,
sum(tot_order) total_order,
round(sum(tot_revenue),2) total_revenue
from total_order_cust_state
group by volume_segment;